In [1]:
import numpy as np
from sklearn.utils import shuffle
import matplotlib.pyplot as plt
from sklearn.decomposition import PCA

X_speech = np.load(r"E:\Pythonfile\Voice-Activity-Detect\data/feature/train/stft_lbp_speech.npy")
X_non_1 = np.load(r"E:\Pythonfile\Voice-Activity-Detect\data/feature/train/stft_lbp_nonspeech_1.npy")
X_non_2 = np.load(r"E:\Pythonfile\Voice-Activity-Detect\data/feature/train/stft_lbp_nonspeech_2.npy")
X_non = np.concatenate([X_non_1, X_non_2], axis=0)

X = np.vstack([X_non, X_speech]).astype(np.float32)
y = np.hstack([
    np.zeros(len(X_non), dtype=np.int8),
    np.ones(len(X_speech), dtype=np.int8)
])

X, y = shuffle(X, y, random_state=42)
print("Final shape:", X.shape, y.shape)

# # ===== PCA 3D =====
# pca = PCA(n_components=3)
# X_3d = pca.fit_transform(X)

# # ===== Plot 3D =====
# fig = plt.figure(figsize=(8,6))
# ax = fig.add_subplot(111, projection='3d')

# ax.scatter(X_3d[y==0,0], X_3d[y==0,1], X_3d[y==0,2],
#            s=5, alpha=0.5, label='Non-speech')

# ax.scatter(X_3d[y==1,0], X_3d[y==1,1], X_3d[y==1,2],
#            s=5, alpha=0.5, label='Speech')

# # ===== chỉnh góc xoay ở đây =====
# ax.view_init(elev=10, azim=10)

# ax.set_title("MFCC 3D Visualization")
# ax.set_xlabel("PC1")
# ax.set_ylabel("PC2")
# ax.set_zlabel("PC3")

# ax.legend()
# plt.show()

Final shape: (126708, 256) (126708,)


In [2]:
import numpy as np
from xgboost import XGBClassifier
from sklearn.model_selection import StratifiedKFold
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score
from sklearn.metrics import roc_curve


# ===== Hàm tính EER =====
def compute_eer(y_true, y_score):
    fpr, tpr, thresholds = roc_curve(y_true, y_score)
    fnr = 1 - tpr
    eer = fpr[np.nanargmin(np.absolute((fnr - fpr)))]
    return eer

spoof = np.sum(y == 0)
bonafide = np.sum(y == 1)

scale_pos_weight = spoof / bonafide

print("scale_pos_weight:", scale_pos_weight)


# ===== 5 Fold CV =====
kf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

accs = []
precisions = []
recalls = []
f1s = []
aucs = []
eers = []

count = 1

for train_idx, test_idx in kf.split(X, y):

    print(f"Training phase: {count}")
    count += 1

    X_train, X_test = X[train_idx], X[test_idx]
    y_train, y_test = y[train_idx], y[test_idx]

    model = Pipeline([
        ("scaler", StandardScaler()),
        ("xgb", XGBClassifier(
            n_estimators=300,
            max_depth=6,
            learning_rate=0.1,
            subsample=0.8,
            colsample_bytree=0.8,
            scale_pos_weight=scale_pos_weight,
            eval_metric="logloss",
            n_jobs=-1,
            random_state=42
        ))
    ])

    model.fit(X_train, y_train)

    y_pred = model.predict(X_test)
    y_score = model.predict_proba(X_test)[:, 1]

    accs.append(accuracy_score(y_test, y_pred))
    precisions.append(precision_score(y_test, y_pred))
    recalls.append(recall_score(y_test, y_pred))
    f1s.append(f1_score(y_test, y_pred))
    aucs.append(roc_auc_score(y_test, y_score))
    eers.append(compute_eer(y_test, y_score))

    # print(f"Accuracy: {accs[-1]}")
    # print(f"Precision: {precisions[-1]}")
    # print(f"Recall: {recalls[-1]}")
    # print(f"F1: {f1s[-1]}")
    # print(f"EER: {eers[-1]}")
    # print()


print("===== 5 Fold CV Result =====")
print("Accuracy :", np.mean(accs))
print("Precision:", np.mean(precisions))
print("Recall   :", np.mean(recalls))
print("F1-score :", np.mean(f1s))
print("ROC-AUC  :", np.mean(aucs))
print("EER      :", np.mean(eers))

scale_pos_weight: 0.9792867519565116
Training phase: 1
Training phase: 2
Training phase: 3
Training phase: 4
Training phase: 5
===== 5 Fold CV Result =====
Accuracy : 0.9973719093457747
Precision: 0.9972205294498204
Recall   : 0.9975787608159532
F1-score : 0.9973995767319861
ROC-AUC  : 0.9999415668798068
EER      : 0.0025840984041737456


In [3]:
from sklearn.ensemble import RandomForestClassifier

spoof = np.sum(y == 0)
bonafide = np.sum(y == 1)

print("spoof:", spoof)
print("bonafide:", bonafide)


# ===== 5 Fold CV =====
kf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

accs = []
precisions = []
recalls = []
f1s = []
aucs = []
eers = []

count = 1

for train_idx, test_idx in kf.split(X, y):

    print(f"Training phase: {count}")
    count += 1

    X_train, X_test = X[train_idx], X[test_idx]
    y_train, y_test = y[train_idx], y[test_idx]

    model = Pipeline([
        ("scaler", StandardScaler()),
        ("rf", RandomForestClassifier(
            n_estimators=300,
            max_depth=None,
            class_weight="balanced",
            n_jobs=-1,
            random_state=42
        ))
    ])

    model.fit(X_train, y_train)

    y_pred = model.predict(X_test)
    y_score = model.predict_proba(X_test)[:, 1]

    accs.append(accuracy_score(y_test, y_pred))
    precisions.append(precision_score(y_test, y_pred))
    recalls.append(recall_score(y_test, y_pred))
    f1s.append(f1_score(y_test, y_pred))
    aucs.append(roc_auc_score(y_test, y_score))
    eers.append(compute_eer(y_test, y_score))

    # print(f"Accuracy: {accs[-1]}")
    # print(f"Precision: {precisions[-1]}")
    # print(f"Recall: {recalls[-1]}")
    # print(f"F1: {f1s[-1]}")
    # print(f"EER: {eers[-1]}")
    # print()


print("===== 5 Fold CV Result =====")
print("Accuracy :", np.mean(accs))
print("Precision:", np.mean(precisions))
print("Recall   :", np.mean(recalls))
print("F1-score :", np.mean(f1s))
print("ROC-AUC  :", np.mean(aucs))
print("EER      :", np.mean(eers))

spoof: 62691
bonafide: 64017
Training phase: 1
Training phase: 2
Training phase: 3
Training phase: 4
Training phase: 5
===== 5 Fold CV Result =====
Accuracy : 0.9937415156748962
Precision: 0.9940767039004413
Recall   : 0.9935329402231489
F1-score : 0.9938044372580176
ROC-AUC  : 0.9997594584888227
EER      : 0.0063166798993743445


In [ ]:
from sklearn.model_selection import StratifiedKFold
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score,
    f1_score, roc_auc_score
)

import numpy as np

# ===============================
# Thống kê dữ liệu
# ===============================
spoof = np.sum(y == 0)
bonafide = np.sum(y == 1)

print("spoof:", spoof)
print("bonafide:", bonafide)

# ===============================
# 5 Fold CV
# ===============================
kf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)


# =====================================================
# Hàm dùng chung để train và evaluate
# =====================================================
def evaluate_model(model, model_name):
    accs = []
    precisions = []
    recalls = []
    f1s = []
    aucs = []
    eers = []

    count = 1

    for train_idx, test_idx in kf.split(X, y):

        print(f"{model_name} - Training phase: {count}")
        count += 1

        X_train, X_test = X[train_idx], X[test_idx]
        y_train, y_test = y[train_idx], y[test_idx]

        model.fit(X_train, y_train)

        y_pred = model.predict(X_test)

        # Nếu model có predict_proba
        if hasattr(model, "predict_proba"):
            y_score = model.predict_proba(X_test)[:, 1]
        else:
            y_score = model.decision_function(X_test)

        accs.append(accuracy_score(y_test, y_pred))
        precisions.append(precision_score(y_test, y_pred))
        recalls.append(recall_score(y_test, y_pred))
        f1s.append(f1_score(y_test, y_pred))
        aucs.append(roc_auc_score(y_test, y_score))
        eers.append(compute_eer(y_test, y_score))

    print(f"\n===== {model_name} Result =====")
    print("Accuracy :", np.mean(accs))
    print("Precision:", np.mean(precisions))
    print("Recall   :", np.mean(recalls))
    print("F1-score :", np.mean(f1s))
    print("ROC-AUC  :", np.mean(aucs))
    print("EER      :", np.mean(eers))
    print("\n")


# =====================================================
# 1. Logistic Regression
# =====================================================
from sklearn.linear_model import LogisticRegression

log_model = Pipeline([
    ("scaler", StandardScaler()),
    ("clf", LogisticRegression(
        max_iter=3000,
        class_weight="balanced",
        random_state=42
    ))
])

evaluate_model(log_model, "Logistic Regression")


# =====================================================
# 2. KNN
# =====================================================
from sklearn.neighbors import KNeighborsClassifier

knn_model = Pipeline([
    ("scaler", StandardScaler()),
    ("clf", KNeighborsClassifier(
        n_neighbors=5,
        weights="distance"
    ))
])

evaluate_model(knn_model, "KNN")


# =====================================================
# 3. SVM
# =====================================================
from sklearn.svm import SVC

svm_model = Pipeline([
    ("scaler", StandardScaler()),
    ("clf", SVC(
        kernel="linear",
        C=1,
        probability=True,
        class_weight="balanced",
        random_state=42
    ))
])

evaluate_model(svm_model, "SVM")

spoof: 62691
bonafide: 64017
Logistic Regression - Training phase: 1
Logistic Regression - Training phase: 2
Logistic Regression - Training phase: 3
Logistic Regression - Training phase: 4
Logistic Regression - Training phase: 5

===== Logistic Regression Result =====
Accuracy : 0.9908924511008335
Precision: 0.9897784632217768
Recall   : 0.9922207953496528
F1-score : 0.9909979532459889
ROC-AUC  : 0.9993254132145672
EER      : 0.009267656613392058


KNN - Training phase: 1
KNN - Training phase: 2
KNN - Training phase: 3
KNN - Training phase: 4
KNN - Training phase: 5

===== KNN Result =====
Accuracy : 0.9900479844628057
Precision: 0.9873723102452366
Recall   : 0.9930018525268027
F1-score : 0.9901789844366924
ROC-AUC  : 0.997202750705191
EER      : 0.009985485896540677


SVM - Training phase: 1
